In [1]:
# for working part
import numpy as np
import pandas as pd



#  for ml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score,recall_score,f1_score,roc_auc_score

In [2]:
# reading the parquet file created in notebook 2

lead_conversion_data = pd.read_parquet("C:\\Users\\hp5cd\\OneDrive\\Desktop\\Python\\lead-conversion-prediction\\data\\processed_data.parquet")

In [3]:
# hume neeche model train krne mein errors aaye toh humne ye kiya hai again aur phir se re run krne ja rhe hai codes

lead_conversion_data['inbound_outbound_ratio'] = lead_conversion_data['inbound_outbound_ratio'].replace([np.inf,-np.inf],np.nan).fillna(0)

lead_conversion_data.fillna(0,inplace=True)

lead_conversion_data['profile'] = lead_conversion_data['profile'].replace(0,"unknown_profile")

In [4]:
# extracting important columns for this model
# as this will be used when creating leads we will have columns only that can be there after creating leads (x1)
# and when assigned we will have the columns that are in x2 


x1 = lead_conversion_data[['lead_source']]
x2 = lead_conversion_data[['lead_source','owner','assigned_month','assigned_year']]
x3 = lead_conversion_data[['owner', 'lead_source', 'profile', 'total_duration', 'call_count',
       'distinct_call_days', 'connected_call_count', 'missed_call_count',
       'inbound_call_count', 'outbound_call_count',
       'assigned_month', 'assigned_year', 'followup_done', 'average_duration',
       'connection_rate', 'miss_rate', 'average_call_per_day',
       'average_duration_per_day', 'inbound_outbound_ratio',
       'time_taken_for_first_touch', 'call_span_days', 'call_frequency']]


y = lead_conversion_data['converted']


In [5]:
# splitting again for the three models

x1_train,x1_test,y1_train,y1_test = train_test_split(x1,y,random_state=42,stratify=y,test_size=0.2)

x2_train,x2_test,y2_train,y2_test = train_test_split(x2,y,random_state=42,stratify=y,test_size=0.2)

x3_train,x3_test,y3_train,y3_test = train_test_split(x3,y,random_state=42,stratify=y,test_size=0.2)

In [6]:
#  columns seperation

categorical_x1 = ['lead_source']

categorical_x2 = ['lead_source','owner','assigned_month','assigned_year']

categorical_x3 = ['owner', 'lead_source', 'profile','assigned_month',
                   'assigned_year']

numerical_x3 = ['total_duration', 'call_count','distinct_call_days',
                 'connected_call_count', 'missed_call_count',
                 'inbound_call_count', 'outbound_call_count','average_duration',
                 'connection_rate', 'miss_rate', 'average_call_per_day',
                 'average_duration_per_day', 'inbound_outbound_ratio',
                 'time_taken_for_first_touch', 'call_span_days', 'call_frequency']

binary_x3 = ['followup_done']

In [7]:
def preprocessor(categorical_cols,numerical_cols = None, binary_cols = None):
    transformers = []

    if categorical_cols:
        transformers.append(('cat',OneHotEncoder(handle_unknown='ignore'), categorical_cols))


    elif numerical_cols:
        transformers.append(('num',StandardScaler(),numerical_cols))


    elif binary_cols:
        transformers.append(('binary','passthrough',binary_cols))

    preprocessor = ColumnTransformer(transformers=transformers)


    return preprocessor

In [9]:
def models(class_weight = None, scale_pos_weight = 1, rf_estimators = 20, rf_max_depth = None,
            xgb_estimators = 50, xgb_max_depth = 5, xgb_lr = 0.1):
    
    models = {'LR':LogisticRegression(max_iter=1000,class_weight=class_weight),
              
              'RF':RandomForestClassifier(n_estimators=rf_estimators,max_depth=rf_max_depth,n_jobs=-1,
                                              random_state=42,class_weight=class_weight),

                'XGB':XGBClassifier(n_estimators=xgb_estimators,max_depth=xgb_max_depth,learning_rate=xgb_lr,
                                      n_jobs=-1,random_state=42,scale_pos_weight = scale_pos_weight)}
    

    return models

In [ ]:
def evaluate (preprocessor,models,x_train,x_test,y_train,y_test):
    